# Clinical Risk-Score Benchmarks

Compares USPSTF eligibility, published PLCOm2012 inference, and retrained clinical-variable models on the cohort saved by Notebook 01. All learned predictions are out of fold using the shared fold assignments saved by this notebook.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_DIR = Path(
    os.environ.get("LUNG_CANCER_DATA_DIR", Path.home() / "lungcancer")
).expanduser()
RUN_DIR = DATA_DIR / "paper_reproduction"
PRED_DIR = RUN_DIR / "predictions"
FIG_DIR = RUN_DIR / "figures"
for directory in [RUN_DIR, PRED_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

COHORT_PATH = DATA_DIR / "paper_cohort.parquet"
if not COHORT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {COHORT_PATH}. Run 01_Cohort_Construction.ipynb first."
    )

cohort = pd.read_parquet(COHORT_PATH).reset_index(drop=True)
cohort["person_id"] = pd.to_numeric(cohort["person_id"], errors="raise").astype("int64")
cohort["has_lungcancer_3Y"] = pd.to_numeric(
    cohort["has_lungcancer_3Y"], errors="raise"
).astype(int)

print("Cohort:", cohort.shape)
print("Cases:", int(cohort["has_lungcancer_3Y"].sum()))

## Recover education survey data

If the reconstructed cohort contains only the literal value `Missing` for education, retrieve the original education response from `ds_survey` (question concept ID `1585940`) and merge it by `person_id`. This avoids rerunning the full cohort reconstruction.


In [ ]:
education_missing_labels = {
    "", "missing", "nan", "none", "pmi: skip", "pmi: prefer not to answer"
}

if "education_group" in cohort.columns:
    education_text = cohort["education_group"].astype("string").str.strip().str.lower()
    education_unavailable = education_text.isna() | education_text.isin(education_missing_labels)
else:
    education_unavailable = pd.Series(True, index=cohort.index)

if education_unavailable.all():
    if "WORKSPACE_CDR" not in os.environ:
        raise RuntimeError(
            "Education is unavailable and WORKSPACE_CDR is not set. "
            "Run this notebook in the All of Us Workbench."
        )

    education_sql = f''' 
        SELECT person_id, answer
        FROM `{os.environ["WORKSPACE_CDR"]}.ds_survey`
        WHERE question_concept_id = 1585940
    '''
    education_raw = pd.read_gbq(
        education_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )

    education_map = {
        "Highest Grade: Never Attended": "Less than high school",
        "Highest Grade: One Through Four": "Less than high school",
        "Highest Grade: Five Through Eight": "Less than high school",
        "Highest Grade: Nine Through Eleven": "Less than high school",
        "Highest Grade: Twelve Or GED": "High school or GED",
        "Highest Grade: College One to Three": "Some college",
        "Highest Grade: College Graduate": "College graduate",
        "Highest Grade: Advanced Degree": "Advanced degree",
        "PMI: Skip": "Missing",
        "PMI: Prefer Not To Answer": "Missing",
    }
    education_raw["education_group_recovered"] = education_raw["answer"].map(education_map)

    unexpected = education_raw.loc[
        education_raw["answer"].notna()
        & education_raw["education_group_recovered"].isna(),
        "answer",
    ].value_counts()
    if len(unexpected):
        print("Unmapped education responses:")
        print(unexpected)

    education_raw["missing_response"] = (
        education_raw["education_group_recovered"].isna()
        | education_raw["education_group_recovered"].eq("Missing")
    )
    education = (
        education_raw
        .sort_values(["person_id", "missing_response"], ascending=[True, True])
        .drop_duplicates("person_id", keep="first")
        [["person_id", "education_group_recovered"]]
    )

    cohort = cohort.drop(columns=["education_group"], errors="ignore")
    cohort = cohort.merge(education, on="person_id", how="left")
    cohort = cohort.rename(columns={"education_group_recovered": "education_group"})
    cohort["education_group"] = cohort["education_group"].fillna("Missing")

education_text = cohort["education_group"].astype("string").str.strip().str.lower()
usable_education = education_text.notna() & ~education_text.isin(education_missing_labels)
print(cohort["education_group"].value_counts(dropna=False))
print(f"Usable education: {int(usable_education.sum()):,}/{len(cohort):,} "
      f"({usable_education.mean():.1%})")

if not usable_education.any():
    raise RuntimeError("Education recovery produced no usable values.")


In [ ]:
from sklearn.model_selection import StratifiedKFold

FOLD_PATH = RUN_DIR / "cv_folds.parquet"

def get_or_create_folds(df, n_splits=5, seed=42):
    expected = df[["person_id", "has_lungcancer_3Y"]].copy()
    if FOLD_PATH.exists():
        folds = pd.read_parquet(FOLD_PATH)
        check = expected.merge(folds, on="person_id", how="left", validate="one_to_one")
        if check["fold"].isna().any():
            raise ValueError("Saved fold file does not match the current cohort.")
        return check[["person_id", "fold"]]

    y = expected["has_lungcancer_3Y"]
    fold = np.full(len(expected), -1, dtype=int)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold_id, (_, test_idx) in enumerate(splitter.split(expected, y)):
        fold[test_idx] = fold_id
    folds = pd.DataFrame({"person_id": expected["person_id"], "fold": fold})
    folds.to_parquet(FOLD_PATH, index=False)
    return folds

folds = get_or_create_folds(cohort)
cohort = cohort.merge(folds, on="person_id", how="left", validate="one_to_one")
print(cohort.groupby("fold")["has_lungcancer_3Y"].agg(["count", "sum"]))

## USPSTF eligibility sensitivity

Apply the 2021 USPSTF lung-cancer screening eligibility criteria: age 50–80 years, at least 20 pack-years, and current smoking or quitting within 15 years. Sensitivity is the proportion of complete-case incident lung-cancer cases meeting the eligibility rule. Here, a positive classification means **screening eligible**, not a positive LDCT result. Exclusion reasons may overlap.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

df = cohort.copy()
def numeric_column(frame, name):
    """Return a NumPy float64 Series even when an optional column is absent."""
    if name not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype="float64", name=name)
    values = pd.to_numeric(frame[name], errors="coerce")
    return pd.Series(
        values.to_numpy(dtype="float64", na_value=np.nan),
        index=frame.index,
        name=name,
    )


age = numeric_column(df, "age_at_survey_int")
smoking_years = numeric_column(df, "Smoking: Number Of Years")
average_daily = numeric_column(df, "Smoking: Average Daily Cigarette Number")
current_daily = numeric_column(df, "Smoking: Current Daily Cigarette Number")
quit_age = numeric_column(df, "Attempt Quit Smoking: Completely Quit Age")
status = df["smoking_status"].astype("string").str.strip().str.lower()

# Prefer lifetime/average smoking intensity. Current daily cigarettes are a
# fallback only for current smokers; using zero current cigarettes for former
# smokers would incorrectly erase their pack-year history.
cigs_per_day = average_daily.copy()
current_fallback = status.eq("current") & cigs_per_day.isna()
cigs_per_day.loc[current_fallback] = current_daily.loc[current_fallback]

years_since_quit = pd.Series(np.nan, index=df.index, dtype=float)
years_since_quit.loc[status.eq("current")] = 0.0
former = status.eq("former")
years_since_quit.loc[former] = age.loc[former] - quit_age.loc[former]
years_since_quit = years_since_quit.mask(years_since_quit < 0)

pack_years = (cigs_per_day / 20.0) * smoking_years
complete = (
    age.notna() & smoking_years.notna() & cigs_per_day.notna()
    & status.isin(["current", "former"])
    & (~former | years_since_quit.notna())
)

eligible = (
    age.between(50, 80, inclusive="both")
    & pack_years.ge(20)
    & (status.eq("current") | (former & years_since_quit.le(15)))
)

if int(complete.sum()) == 0:
    raise RuntimeError(
        "No participants have complete USPSTF inputs. The saved paper_cohort.parquet "
        "is missing lifetime cigarettes/day and/or quit age. Run the updated "
        "01_Cohort_Construction.ipynb from the BigQuery rebuild cell, then rerun this notebook."
    )

uspstf = pd.DataFrame({
    "person_id": df.loc[complete, "person_id"],
    "y_true": df.loc[complete, "has_lungcancer_3Y"],
    "prob": eligible.loc[complete].astype(float),
    "model": "USPSTF_eligibility",
})
uspstf.to_parquet(PRED_DIR / "USPSTF_eligibility_predictions.parquet", index=False)

# Complete-case confusion matrix. Here, "positive" means USPSTF
# eligible; it does not mean a positive LDCT result.
uspstf_y = uspstf["y_true"].astype(int)
uspstf_selected = uspstf["prob"].astype(bool)

tp = int((uspstf_selected & uspstf_y.eq(1)).sum())
fp = int((uspstf_selected & uspstf_y.eq(0)).sum())
fn = int((~uspstf_selected & uspstf_y.eq(1)).sum())
tn = int((~uspstf_selected & uspstf_y.eq(0)).sum())

complete_n = len(uspstf)
complete_cases = int(uspstf_y.sum())
all_cases = int(df["has_lungcancer_3Y"].sum())
eligible_n = int(uspstf_selected.sum())

def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator else np.nan


uspstf_confusion_matrix = pd.DataFrame(
    [
        [tp, fp, tp + fp],
        [fn, tn, fn + tn],
        [tp + fn, fp + tn, complete_n],
    ],
    index=["USPSTF eligible", "Not eligible", "Total"],
    columns=["Lung cancer", "No lung cancer", "Total"],
)

uspstf_complete_case_metrics = pd.DataFrame([{
    "Full cohort N": len(df),
    "Full cohort cases": all_cases,
    "Complete-case N": complete_n,
    "Complete-case proportion": safe_ratio(complete_n, len(df)),
    "Complete-case cancers": complete_cases,
    "Complete-case prevalence": safe_ratio(complete_cases, complete_n),
    "USPSTF eligible": eligible_n,
    "Eligibility rate": safe_ratio(eligible_n, complete_n),
    "True positives": tp,
    "False positives": fp,
    "False negatives": fn,
    "True negatives": tn,
    "Sensitivity": safe_ratio(tp, tp + fn),
    "Specificity": safe_ratio(tn, tn + fp),
    "PPV of eligibility": safe_ratio(tp, tp + fp),
    "NPV": safe_ratio(tn, tn + fn),
    "Eligible per observed case": safe_ratio(tp + fp, tp),
}])

print("USPSTF complete-case confusion matrix")
display(uspstf_confusion_matrix)

print("USPSTF complete-case metrics")
display(uspstf_complete_case_metrics.T.rename(columns={0: "Value"}))

print("Input completeness audit")
print("Missing average/usable cigarettes per day:", int(cigs_per_day.isna().sum()))
print("Missing smoking years:", int(smoking_years.isna().sum()))
print("Former smokers missing quit age:", int((former & quit_age.isna()).sum()))

# Report why complete-case incident cancers did not meet the eligibility rule.
# Reasons overlap, so the rows should not be summed.
complete_case_cancers = complete & df["has_lungcancer_3Y"].eq(1)
ineligible_cancers = complete_case_cancers & ~eligible
criterion_flags = {
    "Outside age 50–80 years": ~age.between(50, 80, inclusive="both"),
    "Below 20 pack-years": ~pack_years.ge(20),
    "Does not meet current/recent-smoking criterion": ~(
        status.eq("current") | (former & years_since_quit.le(15))
    ),
}

uspstf_case_exclusions = pd.DataFrame([
    {
        "Exclusion reason": reason,
        "Cases": int((ineligible_cancers & failed).sum()),
        "% of ineligible cases": safe_ratio(
            int((ineligible_cancers & failed).sum()), fn
        ),
        "% of complete-case cancers": safe_ratio(
            int((ineligible_cancers & failed).sum()), complete_cases
        ),
    }
    for reason, failed in criterion_flags.items()
]).set_index("Exclusion reason")

print("Overlapping reasons incident cancers were not USPSTF eligible")
display(
    uspstf_case_exclusions.style.format({
        "% of ineligible cases": "{:.1%}",
        "% of complete-case cancers": "{:.1%}",
    })
)

uspstf_confusion_matrix.to_csv(RUN_DIR / "uspstf_complete_case_confusion_matrix.csv")
uspstf_complete_case_metrics.to_csv(
    RUN_DIR / "uspstf_complete_case_metrics.csv", index=False
)
uspstf_case_exclusions.to_csv(RUN_DIR / "uspstf_case_exclusion_reasons.csv")


## Published PLCOm2012 inference

In [ ]:
def mode_or(series, fallback):
    observed = series.dropna()
    return observed.mode().iloc[0] if len(observed) else fallback

df = cohort.copy()
df["smoking_status_clean"] = (
    df["smoking_status"].astype("string").str.strip().str.lower()
)

# Published PLCOm2012 requires current/former smoking status. Missing status is
# treated as unevaluable rather than imputed to current or former.
df = df.loc[df["smoking_status_clean"].isin(["current", "former"])].copy()

edu_map = {
    "Less than high school": 1,
    "High school or GED": 2,
    "Some college": 4,
    "College graduate": 5,
    "Advanced degree": 6,
}

age = pd.to_numeric(df["age_at_survey_int"], errors="coerce").fillna(62)
bmi = pd.to_numeric(df["bmi"], errors="coerce").fillna(27)
smoking_years = pd.to_numeric(
    df["Smoking: Number Of Years"], errors="coerce"
).fillna(27)
quit_years = pd.to_numeric(df["duration_of_quitting"], errors="coerce")
quit_years = quit_years.where(~df["smoking_status_clean"].eq("current"), 0.0).fillna(10)

average_daily = pd.to_numeric(
    df["Smoking: Average Daily Cigarette Number"], errors="coerce"
)
current_daily = pd.to_numeric(
    df["Smoking: Current Daily Cigarette Number"], errors="coerce"
)

# PLCOm2012 smoking intensity represents usual cigarettes/day. Current daily
# cigarettes are often zero for former smokers, so use average/lifetime daily
# cigarettes for both groups and current daily only as a fallback for current
# smokers. Treat zero as missing rather than as one cigarette/day.
cigs = average_daily.mask(average_daily <= 0)
current_fallback = df["smoking_status_clean"].eq("current") & cigs.isna()
valid_current_daily = current_daily.mask(current_daily <= 0)
cigs.loc[current_fallback] = valid_current_daily.loc[current_fallback]
cigarette_median = cigs.dropna().median()
cigs = cigs.fillna(cigarette_median)

print("Cigarettes/day median used for imputation:", cigarette_median)
display(
    pd.DataFrame({
        "status": df["smoking_status_clean"],
        "cigs_per_day": cigs,
    }).groupby("status")["cigs_per_day"].describe()
)

copd = pd.to_numeric(df["COPD_pre"], errors="coerce")
copd = copd.fillna(mode_or(copd, 0))
personal_cancer = pd.to_numeric(df["cancer_history"], errors="coerce")
personal_cancer = personal_cancer.fillna(mode_or(personal_cancer, 0))
family_lung = pd.to_numeric(df["family_lung"], errors="coerce")
family_lung = family_lung.fillna(mode_or(family_lung, 0))

education = df["education_group"].map(edu_map)
missing_education = int(education.isna().sum())
education = education.fillna(4)
print(f"Education assigned reference level 4: {missing_education:,}")

race = df["race_ethnicity"].replace("Missing", pd.NA)
race = race.fillna(mode_or(race, "White")).astype("string").str.strip().str.lower()

current_smoker = df["smoking_status_clean"].eq("current").astype(int)
intensity = (10 / cigs) - 0.4021541613

# Published PLCOm2012 coefficients with the required centering constants.
xb = (
    0.0778868 * (age - 62)
    + 0.3944778 * race.str.contains("black", na=False).astype(int)
    - 0.7434744 * race.str.contains("hispanic", na=False).astype(int)
    - 0.4665850 * race.eq("asian").astype(int)
    + 1.0271520 * race.str.contains(
        "native hawaiian|pacific island", regex=True, na=False
    ).astype(int)
    - 0.0812744 * (education - 4)
    - 0.0274194 * (bmi - 27)
    + 0.3553063 * copd
    + 0.4589971 * personal_cancer
    + 0.5871850 * family_lung
    + 0.2597431 * current_smoker
    - 1.8226060 * intensity
    + 0.0317321 * (smoking_years - 27)
    - 0.0308572 * (quit_years - 10)
    - 4.532506
)

# Design matrix used for a true cohort-specific logistic refit. These are the
# same terms and transformations as the published PLCOm2012 linear predictor.
plco_design = pd.DataFrame({
    "person_id": df["person_id"].astype("int64"),
    "y_true": df["has_lungcancer_3Y"].astype(int),
    "fold": df["fold"].astype(int),
    "age_centered": age - 62,
    "race_black": race.str.contains("black", na=False).astype(int),
    "race_hispanic": race.str.contains("hispanic", na=False).astype(int),
    "race_asian": race.eq("asian").astype(int),
    "race_nhpi": race.str.contains(
        "native hawaiian|pacific island", regex=True, na=False
    ).astype(int),
    "education_centered": education - 4,
    "bmi_centered": bmi - 27,
    "copd": copd,
    "personal_cancer": personal_cancer,
    "family_lung": family_lung,
    "current_smoker": current_smoker,
    "smoking_intensity": intensity,
    "smoking_years_centered": smoking_years - 27,
    "quit_years_centered": quit_years - 10,
})

plco = pd.DataFrame({
    "person_id": df["person_id"],
    "y_true": df["has_lungcancer_3Y"].astype(int),
    "prob": 1 / (1 + np.exp(-xb)),
    "model": "PLCOm2012_published",
})
plco["eligible_1_51"] = plco["prob"].ge(0.0151)
plco.to_parquet(PRED_DIR / "PLCOm2012_published_predictions.parquet", index=False)

print(plco["prob"].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]))
print("Published PLCOm2012 AUROC:", roc_auc_score(plco["y_true"], plco["prob"]))
print("Published PLCOm2012 AUPRC:", average_precision_score(plco["y_true"], plco["prob"]))

# Evaluate the validated PLCOm2012 screening threshold. The model predicts
# six-year risk, while y_true is incident lung cancer within three years in
# this reconstructed cohort.
plco_y = plco["y_true"].astype(int)
plco_selected_1_51 = plco["eligible_1_51"].astype(bool)

plco_tp = int((plco_selected_1_51 & plco_y.eq(1)).sum())
plco_fp = int((plco_selected_1_51 & plco_y.eq(0)).sum())
plco_fn = int((~plco_selected_1_51 & plco_y.eq(1)).sum())
plco_tn = int((~plco_selected_1_51 & plco_y.eq(0)).sum())

def plco_ratio(numerator, denominator):
    return numerator / denominator if denominator else np.nan

plco_1_51_confusion_matrix = pd.DataFrame(
    [
        [plco_tp, plco_fp, plco_tp + plco_fp],
        [plco_fn, plco_tn, plco_fn + plco_tn],
        [plco_tp + plco_fn, plco_fp + plco_tn, len(plco)],
    ],
    index=["PLCOm2012 >=1.51%", "PLCOm2012 <1.51%", "Total"],
    columns=["Lung cancer", "No lung cancer", "Total"],
)

plco_1_51_metrics = pd.DataFrame([{
    "Model": "Published PLCOm2012 (>=1.51%)",
    "N": len(plco),
    "Cases": int(plco_y.sum()),
    "Selected": int(plco_selected_1_51.sum()),
    "Screening capacity": plco_ratio(plco_selected_1_51.sum(), len(plco)),
    "Cases captured": plco_tp,
    "Sensitivity / recall": plco_ratio(plco_tp, plco_tp + plco_fn),
    "Specificity": plco_ratio(plco_tn, plco_tn + plco_fp),
    "PPV / precision": plco_ratio(plco_tp, plco_tp + plco_fp),
    "NPV": plco_ratio(plco_tn, plco_tn + plco_fn),
    "Screened per observed case": (
        plco_ratio(plco_tp + plco_fp, plco_tp)
    ),
}]).set_index("Model")

print("Published PLCOm2012 confusion matrix at the 1.51% threshold")
display(plco_1_51_confusion_matrix)
print("Published PLCOm2012 screening metrics at the 1.51% threshold")
display(plco_1_51_metrics)

plco_1_51_confusion_matrix.to_csv(
    RUN_DIR / "plcom2012_1_51_confusion_matrix.csv"
)
plco_1_51_metrics.to_csv(RUN_DIR / "plcom2012_1_51_metrics.csv")


## All of Us–refitted logistic PLCO

This is a true logistic refit: it preserves the published PLCOm2012 predictor terms and transformations but re-estimates all coefficients within All of Us. Predictions are generated out of fold using the shared five-fold assignments. It is distinct from the nonlinear PLCO-variable XGBoost model.


In [ ]:
from sklearn.linear_model import LogisticRegression

refit_features = [
    "age_centered",
    "race_black",
    "race_hispanic",
    "race_asian",
    "race_nhpi",
    "education_centered",
    "bmi_centered",
    "copd",
    "personal_cancer",
    "family_lung",
    "current_smoker",
    "smoking_intensity",
    "smoking_years_centered",
    "quit_years_centered",
]

X_refit = plco_design[refit_features].astype(float)
y_refit = plco_design["y_true"].astype(int)
fold_refit = plco_design["fold"].astype(int)
oof_logistic = np.zeros(len(plco_design), dtype=float)
refit_fold_rows = []

for fold_id in sorted(fold_refit.unique()):
    test = fold_refit.eq(fold_id).to_numpy()
    train = ~test

    logistic_model = LogisticRegression(
        C=np.inf,
        solver="lbfgs",
        max_iter=2000,
        random_state=42,
    )
    logistic_model.fit(X_refit.loc[train], y_refit.loc[train])
    oof_logistic[test] = logistic_model.predict_proba(X_refit.loc[test])[:, 1]

    fold_auroc = roc_auc_score(y_refit.loc[test], oof_logistic[test])
    fold_auprc = average_precision_score(y_refit.loc[test], oof_logistic[test])
    refit_fold_rows.append({
        "Fold": int(fold_id) + 1,
        "AUROC": fold_auroc,
        "AUPRC": fold_auprc,
    })

    print(
        f"Fold {fold_id + 1}: "
        f"AUROC={fold_auroc:.4f}; "
        f"AUPRC={fold_auprc:.4f}"
    )

refit_fold_metrics = pd.DataFrame(refit_fold_rows).sort_values("Fold").reset_index(drop=True)
refit_fold_summary = refit_fold_metrics[["AUROC", "AUPRC"]].agg(["mean", "std"]).T
print("Refitted logistic five-fold mean ± sample SD")
display(refit_fold_summary)
refit_fold_metrics.to_csv(
    RUN_DIR / "plco_refitted_logistic_fold_metrics.csv",
    index=False,
)

plco_logistic_refit = pd.DataFrame({
    "person_id": plco_design["person_id"],
    "y_true": y_refit,
    "prob": oof_logistic,
    "model": "PLCOm2012_refitted_logistic",
})
plco_logistic_refit.to_parquet(
    PRED_DIR / "PLCOm2012_refitted_logistic_predictions.parquet",
    index=False,
)

print("Refitted logistic AUROC:", roc_auc_score(y_refit, oof_logistic))
print("Refitted logistic AUPRC:", average_precision_score(y_refit, oof_logistic))


## Retrained clinical-variable model

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier

model_df = cohort.copy()
quit_age = pd.to_numeric(model_df.get("Attempt Quit Smoking: Completely Quit Age"), errors="coerce")
age = pd.to_numeric(model_df["age_at_survey_int"], errors="coerce")
status = model_df["smoking_status"].astype("string").str.strip().str.lower()
model_df["years_since_quit"] = (age - quit_age).where(status.eq("former"), 0.0).mask(lambda s: s < 0)

average_daily = pd.to_numeric(
    model_df["Smoking: Average Daily Cigarette Number"], errors="coerce"
)
current_daily = pd.to_numeric(
    model_df["Smoking: Current Daily Cigarette Number"], errors="coerce"
)
status = model_df["smoking_status"].astype("string").str.strip().str.lower()

model_df["cigs_per_day"] = average_daily.mask(average_daily <= 0)
current_fallback = status.eq("current") & model_df["cigs_per_day"].isna()
valid_current_daily = current_daily.mask(current_daily <= 0)
model_df.loc[current_fallback, "cigs_per_day"] = valid_current_daily.loc[current_fallback]




clinical = [
    "education_group", "bmi", "COPD_pre", "family_lung", "years_since_quit",
    "Smoking: Number Of Years", "cigs_per_day",
    "smoking_status", "cancer_history", "age_at_survey_int", "race_ethnicity",
]
X = model_df[clinical].copy()
y = cohort["has_lungcancer_3Y"].astype(int)
cat = ["education_group", "smoking_status", "race_ethnicity"]
num = [c for c in clinical if c not in cat]
for c in num:
    X[c] = pd.to_numeric(X[c], errors="coerce")

oof = np.zeros(len(cohort))
for fold_id in sorted(cohort["fold"].unique()):
    test = cohort["fold"].eq(fold_id).to_numpy()
    train = ~test
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat),
    ])
    model = Pipeline([("prep", prep), ("model", XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", n_jobs=-1, random_state=42,
    ))])
    model.fit(X.loc[train], y.loc[train])
    oof[test] = model.predict_proba(X.loc[test])[:, 1]
    print(f"Fold {fold_id + 1}: AUROC={roc_auc_score(y.loc[test], oof[test]):.4f}")

clinical_pred = pd.DataFrame({
    "person_id": cohort["person_id"], "y_true": y,
    "prob": oof, "model": "Clinical_retrained",
})
clinical_pred.to_parquet(PRED_DIR / "Clinical_retrained_predictions.parquet", index=False)
print("OOF AUROC:", roc_auc_score(y, oof))
print("OOF AUPRC:", average_precision_score(y, oof))
print("OOF Brier:", brier_score_loss(y, oof))

## Clinician-facing screening metrics

Because the trained XGBoost model has no established probability cutoff, evaluate screening utility by ranking out-of-fold predictions. Report sensitivity (recall), PPV, specificity, and screening yield at fixed population capacities. The primary illustrative operating point is the top 10% of predicted risk. A second analysis selects exactly the same number of participants as USPSTF in the shared evaluable cohort.


In [ ]:
def metrics_at_capacity(predictions, fraction=None, number_selected=None, label=None):
    ranked = predictions[["person_id", "y_true", "prob"]].dropna().copy()
    ranked = ranked.sort_values(["prob", "person_id"], ascending=[False, True]).reset_index(drop=True)

    if number_selected is None:
        number_selected = int(np.floor(len(ranked) * fraction))
    number_selected = max(1, min(int(number_selected), len(ranked)))

    selected = np.zeros(len(ranked), dtype=bool)
    selected[:number_selected] = True
    y = ranked["y_true"].astype(int).to_numpy()

    tp = int((selected & (y == 1)).sum())
    fp = int((selected & (y == 0)).sum())
    fn = int((~selected & (y == 1)).sum())
    tn = int((~selected & (y == 0)).sum())

    return pd.Series({
        "N": len(ranked),
        "Cases": int(y.sum()),
        "Selected": number_selected,
        "Screening capacity": number_selected / len(ranked),
        "Cases captured": tp,
        "Sensitivity / recall": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "PPV / precision": tp / (tp + fp),
        "NPV": tn / (tn + fn),
        "Screened per observed case": number_selected / tp if tp else np.inf,
    }, name=label)


capacity_rows = []
for model_name, predictions in [
    ("Published PLCOm2012", plco[["person_id", "y_true", "prob"]]),
    ("All of Us–refitted logistic PLCO", plco_logistic_refit[["person_id", "y_true", "prob"]]),
    ("PLCO-variable XGBoost", clinical_pred[["person_id", "y_true", "prob"]]),
]:
    for fraction in [0.05, 0.10, 0.20]:
        capacity_label = f"Top {int(fraction * 100)}%"
        row = metrics_at_capacity(
            predictions,
            fraction=fraction,
            label=capacity_label,
        ).to_dict()
        row["Model"] = model_name
        row["Capacity"] = capacity_label
        capacity_rows.append(row)

capacity_results = (
    pd.DataFrame(capacity_rows)
    .set_index(["Model", "Capacity"])
)

print("Fixed-capacity screening performance")
display(capacity_results)

slide_metrics = capacity_results[[
    "Cases captured",
    "Sensitivity / recall",
    "Specificity",
    "PPV / precision",
    "Screened per observed case",
]].copy()

print("Slide-ready clinical metrics")
display(slide_metrics)

print("Primary clinician-facing operating point: top 10%")
display(capacity_results.xs("Top 10%", level="Capacity"))


In [ ]:
# Match the number selected by USPSTF on the same evaluable participants.
shared = uspstf[["person_id", "y_true", "prob"]].rename(
    columns={"prob": "uspstf_selected"}
).merge(
    clinical_pred[["person_id", "prob"]].rename(columns={"prob": "xgb_prob"}),
    on="person_id",
    how="inner",
    validate="one_to_one",
)

uspstf_for_metrics = shared[["person_id", "y_true"]].copy()
uspstf_for_metrics["prob"] = shared["uspstf_selected"].astype(float)

def metrics_for_binary_rule(predictions, selected_column, label):
    selected = predictions[selected_column].astype(bool).to_numpy()
    y = predictions["y_true"].astype(int).to_numpy()
    tp = int((selected & (y == 1)).sum())
    fp = int((selected & (y == 0)).sum())
    fn = int((~selected & (y == 1)).sum())
    tn = int((~selected & (y == 0)).sum())
    return pd.Series({
        "N": len(y),
        "Cases": int(y.sum()),
        "Selected": int(selected.sum()),
        "Screening capacity": selected.mean(),
        "Cases captured": tp,
        "Sensitivity / recall": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "PPV / precision": tp / (tp + fp),
        "NPV": tn / (tn + fn),
        "Screened per observed case": selected.sum() / tp if tp else np.inf,
    }, name=label)

# Head-to-head comparison on the exact USPSTF complete-case cohort. Unlike
# the capacity-matched comparison below, PLCOm2012 uses its validated 1.51%
# six-year-risk threshold, so the two approaches may select different counts.
shared_1_51 = shared[["person_id", "y_true", "uspstf_selected"]].merge(
    plco[["person_id", "eligible_1_51"]].rename(
        columns={"eligible_1_51": "plco_selected_1_51"}
    ),
    on="person_id",
    how="inner",
    validate="one_to_one",
)

uspstf_shared_1_51_metrics = metrics_for_binary_rule(
    shared_1_51,
    "uspstf_selected",
    "USPSTF",
)
plco_shared_1_51_metrics = metrics_for_binary_rule(
    shared_1_51,
    "plco_selected_1_51",
    "Published PLCOm2012 (>=1.51%)",
)
shared_1_51_results = pd.concat(
    [uspstf_shared_1_51_metrics, plco_shared_1_51_metrics],
    axis=1,
).T

print(
    f"Shared-cohort, 1.51% comparison: N={len(shared_1_51):,}; "
    f"cases={int(shared_1_51['y_true'].sum()):,}"
)
display(shared_1_51_results)
shared_1_51_results.to_csv(
    RUN_DIR / "uspstf_vs_plcom2012_1_51_shared_cohort.csv"
)

uspstf_metrics = metrics_for_binary_rule(shared, "uspstf_selected", "USPSTF")

xgb_shared = shared[["person_id", "y_true", "xgb_prob"]].rename(
    columns={"xgb_prob": "prob"}
)
xgb_matched_metrics = metrics_at_capacity(
    xgb_shared,
    number_selected=int(shared["uspstf_selected"].sum()),
    label="PLCO-variable XGBoost: USPSTF-matched capacity",
)

matched_capacity_results = pd.concat(
    [uspstf_metrics, xgb_matched_metrics], axis=1
).T

print("Shared-cohort, equal-screening-capacity comparison")
display(matched_capacity_results)


## Final model-comparison table

All three models are evaluated on the same published-PLCO-evaluable participants. AUROC and AUPRC use continuous out-of-fold predictions for trained models. Sensitivity and PPV are evaluated after selecting the top 5%, 10%, and 20% of each model's risk ranking. Paired DeLong tests compare AUROC only, using PLCO-variable XGBoost as the reference.


In [ ]:
# Evaluate all three PLCO models on one common population.
common_evaluation = plco[["person_id", "y_true"]].copy()

comparison_models = {
    "Published PLCOm2012": plco[["person_id", "prob"]],
    "All of Us–refitted logistic PLCO": plco_logistic_refit[["person_id", "prob"]],
    "PLCO-variable XGBoost": clinical_pred[["person_id", "prob"]],
}

comparison_rows = []
for model_name, predictions in comparison_models.items():
    evaluated = common_evaluation.merge(
        predictions,
        on="person_id",
        how="inner",
        validate="one_to_one",
    )

    if len(evaluated) != len(common_evaluation):
        raise RuntimeError(
            f"{model_name} has {len(evaluated):,} shared rows; "
            f"expected {len(common_evaluation):,}."
        )

    row = {
        "Model": model_name,
        "AUROC": roc_auc_score(evaluated["y_true"], evaluated["prob"]),
        "AUPRC": average_precision_score(evaluated["y_true"], evaluated["prob"]),
    }
    for fraction in [0.05, 0.10, 0.20]:
        percent = int(fraction * 100)
        capacity_metrics = metrics_at_capacity(
            evaluated,
            fraction=fraction,
            label=model_name,
        )
        row[f"Sensitivity@{percent}%"] = capacity_metrics["Sensitivity / recall"]
        row[f"PPV@{percent}%"] = capacity_metrics["PPV / precision"]
    comparison_rows.append(row)

model_comparison = pd.DataFrame(comparison_rows).set_index("Model")

print(
    f"Common evaluation cohort: N={len(common_evaluation):,}; "
    f"cases={int(common_evaluation['y_true'].sum()):,}"
)
display(model_comparison.style.format({
    "AUROC": "{:.3f}",
    "AUPRC": "{:.3f}",
    "Sensitivity@5%": "{:.1%}",
    "PPV@5%": "{:.1%}",
    "Sensitivity@10%": "{:.1%}",
    "PPV@10%": "{:.1%}",
    "Sensitivity@20%": "{:.1%}",
    "PPV@20%": "{:.1%}",
}))

model_comparison.to_csv(RUN_DIR / "plco_model_comparison.csv")


# Paired DeLong tests below compare AUROC (not AUPRC) on the exact same
# participants and out-of-fold predictions, using XGBoost as the reference.
from scipy import stats


def compute_midrank(values):
    values = np.asarray(values)
    order = np.argsort(values)
    sorted_values = values[order]
    midranks = np.zeros(len(values), dtype=float)
    start = 0
    while start < len(values):
        end = start + 1
        while end < len(values) and sorted_values[end] == sorted_values[start]:
            end += 1
        midranks[start:end] = 0.5 * (start + end - 1)
        start = end
    result = np.empty(len(values), dtype=float)
    result[order] = midranks + 1
    return result


def fast_delong(predictions_sorted_transposed, label_1_count):
    m = int(label_1_count)
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    classifier_count = predictions_sorted_transposed.shape[0]

    tx = np.empty((classifier_count, m), dtype=float)
    ty = np.empty((classifier_count, n), dtype=float)
    tz = np.empty((classifier_count, m + n), dtype=float)
    for classifier_index in range(classifier_count):
        tx[classifier_index] = compute_midrank(positive_examples[classifier_index])
        ty[classifier_index] = compute_midrank(negative_examples[classifier_index])
        tz[classifier_index] = compute_midrank(
            predictions_sorted_transposed[classifier_index]
        )

    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1) / (2 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1 - (tz[:, m:] - ty) / m
    covariance = np.cov(v01) / m + np.cov(v10) / n
    return aucs, covariance


def paired_delong_auroc_test(y_true, first_prob, second_prob):
    y_true = np.asarray(y_true, dtype=int)
    first_prob = np.asarray(first_prob, dtype=float)
    second_prob = np.asarray(second_prob, dtype=float)
    if set(np.unique(y_true)) != {0, 1}:
        raise ValueError("DeLong requires a binary outcome containing both classes.")
    if not (len(y_true) == len(first_prob) == len(second_prob)):
        raise ValueError("Outcome and prediction arrays must have equal lengths.")

    order = np.argsort(-y_true, kind="mergesort")
    predictions = np.vstack([first_prob, second_prob])[:, order]
    aucs, covariance = fast_delong(predictions, y_true.sum())
    contrast = np.array([1.0, -1.0])
    variance = float(contrast @ covariance @ contrast.T)
    if variance <= 0:
        if np.isclose(aucs[0], aucs[1]):
            return aucs, 1.0
        raise RuntimeError("DeLong variance was non-positive.")
    z_score = abs(float(aucs[0] - aucs[1])) / np.sqrt(variance)
    return aucs, float(2 * stats.norm.sf(z_score))


probability_columns = {
    "Published PLCOm2012": "published_prob",
    "All of Us–refitted logistic PLCO": "refitted_prob",
    "PLCO-variable XGBoost": "xgb_prob",
}
aligned_predictions = common_evaluation.copy()
for model_name, predictions in comparison_models.items():
    probability_column = probability_columns[model_name]
    aligned_predictions = aligned_predictions.merge(
        predictions.rename(columns={"prob": probability_column}),
        on="person_id",
        how="left",
        validate="one_to_one",
    )
    missing = int(aligned_predictions[probability_column].isna().sum())
    if missing:
        raise RuntimeError(f"{model_name} is missing {missing:,} predictions.")

aligned_predictions = aligned_predictions.merge(
    cohort[["person_id", "fold"]],
    on="person_id",
    how="left",
    validate="one_to_one",
)
if aligned_predictions["fold"].isna().any():
    raise RuntimeError("Fold assignments are missing for the common evaluation cohort.")

reference_model = "PLCO-variable XGBoost"
reference_prob = aligned_predictions[probability_columns[reference_model]].to_numpy()
y_common = aligned_predictions["y_true"].astype(int).to_numpy()

fold_rows = []
delong_rows = []
for model_name, probability_column in probability_columns.items():
    model_prob = aligned_predictions[probability_column].to_numpy()
    for fold_id, fold_data in aligned_predictions.groupby("fold", sort=True):
        fold_rows.append({
            "Model": model_name,
            "Fold": int(fold_id) + 1,
            "AUROC": roc_auc_score(fold_data["y_true"], fold_data[probability_column]),
            "AUPRC": average_precision_score(
                fold_data["y_true"], fold_data[probability_column]
            ),
        })

    if model_name == reference_model:
        p_value = np.nan
        model_auc = roc_auc_score(y_common, model_prob)
        reference_auc = model_auc
    else:
        aucs, p_value = paired_delong_auroc_test(
            y_common, model_prob, reference_prob
        )
        model_auc, reference_auc = aucs

    delong_rows.append({
        "Model": model_name,
        "Reference": reference_model,
        "Model AUROC": model_auc,
        "Reference AUROC": reference_auc,
        "AUROC difference": model_auc - reference_auc,
        "DeLong p-value": p_value,
    })

fold_model_metrics = pd.DataFrame(fold_rows)
fold_model_summary = (
    fold_model_metrics.groupby("Model")[["AUROC", "AUPRC"]]
    .agg(["mean", "std"])
)
delong_results = pd.DataFrame(delong_rows).set_index("Model")
model_comparison["DeLong p-value vs PLCO-variable XGBoost"] = (
    delong_results["DeLong p-value"]
)

paper_rows = []
for model_name, row in model_comparison.iterrows():
    fold_summary = fold_model_summary.loc[model_name]
    if model_name == "Published PLCOm2012":
        auroc_text = f"{row['AUROC']:.3f}"
        auprc_text = f"{row['AUPRC']:.3f}"
    else:
        auroc_text = (
            f"{fold_summary[('AUROC', 'mean')]:.3f} ± "
            f"{fold_summary[('AUROC', 'std')]:.3f}"
        )
        auprc_text = (
            f"{fold_summary[('AUPRC', 'mean')]:.3f} ± "
            f"{fold_summary[('AUPRC', 'std')]:.3f}"
        )

    p_value = row["DeLong p-value vs PLCO-variable XGBoost"]
    if pd.isna(p_value):
        p_value_text = "Reference"
    elif p_value < 0.001:
        p_value_text = "<0.001"
    else:
        p_value_text = f"{p_value:.3f}"

    paper_rows.append({
        "Model": model_name,
        "AUROC (mean ± SD)": auroc_text,
        "AUPRC (mean ± SD)": auprc_text,
        "Top 5% Sens./PPV": f"{row['Sensitivity@5%']:.1%} / {row['PPV@5%']:.1%}",
        "Top 10% Sens./PPV": f"{row['Sensitivity@10%']:.1%} / {row['PPV@10%']:.1%}",
        "Top 20% Sens./PPV": f"{row['Sensitivity@20%']:.1%} / {row['PPV@20%']:.1%}",
        "DeLong p-value vs PLCO-variable XGBoost": p_value_text,
    })
paper_model_comparison = pd.DataFrame(paper_rows).set_index("Model")

print("Paper-ready model comparison")
display(paper_model_comparison)
print("Paired DeLong AUROC comparisons (XGBoost is the reference)")
display(delong_results)

model_comparison.to_csv(RUN_DIR / "plco_model_comparison.csv")
paper_model_comparison.to_csv(RUN_DIR / "plco_model_comparison_paper_ready.csv")
fold_model_metrics.to_csv(RUN_DIR / "plco_model_comparison_fold_metrics.csv", index=False)
delong_results.to_csv(RUN_DIR / "plco_model_comparison_delong_auroc.csv")
